# Technical Review — From Storage Stress to Cross-Sectional Commodity Factors

**A complete record of what was built, what was learned, and why the strategy changed.**

*Last updated: 2026-08-10*

---

## Executive summary

The project began as a **single-market thesis**: that post-2016 LNG export growth tightened
US natural-gas deliverability, that salt-cavern storage stress measures this tightening, and
that NYMEX calendar spreads had not yet repriced it.

That thesis was **retired**. It failed its own pre-registered week-1 regime gate, and after two
measurement bugs were fixed, no agent beat a properly calibrated random benchmark.

It was replaced by **Pivot A**: a dollar-neutral, cross-sectional long/short commodity
portfolio built on three premia that are documented and replicated across decades of
literature — **carry**, **momentum**, and **basis-momentum**.

| | Original (NG storage stress) | Pivot A (cross-sectional factors) |
|---|---|---|
| Markets | 1 (natural gas) | 23 commodities |
| Instrument | calendar spread | front contract, long/short |
| Signal | Deliverability Stress Index | carry + momentum + basis-momentum |
| Evidence base | one original hypothesis | decades of replicated literature |
| Sample | 6–48 trades | 6,676 days × 23 markets |
| Result | **no edge vs chance** | **Sharpe 0.78** (post-2019, 3× costs, vol-targeted) |

**What survived the pivot is the most valuable thing built: the methodology.** Roll-safe
returns, explicitly charged costs, and calibrated null benchmarking are what exposed the
original result as noise — and they are what makes the new result credible.

---
# Part I — The original strategy

## I.1 The thesis

Three linked claims:

1. **Post-2016 LNG exports tightened the system.** Sabine Pass began exporting Feb 2016; US gas
   demand gained a large, price-inelastic new source.
2. **Salt-cavern utilisation measures the resulting stress.** Salt caverns are *fast-cycling*
   storage — they turn over several times a season, unlike depleted reservoirs (~once a year).
   They are the marginal, short-term balancing supply, so how hard they are working is a
   deliverability-stress indicator.
3. **Calendar spreads had not repriced this.** Therefore an edge existed.

The instrument was a **seasonal calendar spread**: front month vs **next January** in injection
season (Apr–Oct), front vs **next April** in withdrawal (Nov–Mar).

## I.2 The Deliverability Stress Index (DSI)

A five-step pipeline turning weekly EIA storage data into a trading signal:

In [ ]:
import numpy as np, pandas as pd

# --- The DSI pipeline, condensed. Each step is one line of economics. ------

def utilization(net_flow_bcf, max_rate_bcf_wk):
    '''Step 1. What fraction of the region's max weekly cycling rate is in use?
    Signed: + on injection, - on withdrawal. Clipped so one bad print can't
    blow up the exponential in step 2.'''
    return np.clip(net_flow_bcf / max_rate_bcf_wk, -1.2, 1.2)

def convex_stress(u, k=4.0):
    '''Step 2. g(u) = sign(u) * (e^{k|u|} - 1)/(e^k - 1).

    WHY CONVEX: a system at 95% utilisation absorbs a shock far worse than one
    at 70%. Stress is NOT proportional to usage - it accelerates as you approach
    the physical limit. Near u=0 this is ~linear; near |u|=1 it explodes.'''
    return np.sign(u) * (np.expm1(k * np.abs(u)) / np.expm1(k))

# The non-linearity is the entire point - see how the last 5% of capacity
# costs more than the first 70%:
for u in (0.30, 0.70, 0.90, 0.95, 1.00):
    print(f"utilisation {u:>5.0%}  ->  stress {convex_stress(u):.3f}")

Steps 3–5 completed the pipeline:

- **Step 3 — smooth.** 3-week EWMA, to remove single-week noise.
- **Step 4 — deseasonalise.** Subtract an *expanding* week-of-year mean (expanding, so no
  look-ahead). The market already prices the ordinary seasonal pattern; only the anomaly is
  tradeable.
- **Step 5 — residualise.** Rolling 3-year OLS of the signal on changes in Waha and Dominion
  South basis. The **residual** is stress *not* already explained by regional price dynamics.

The output, `dsi_clean`, was converted to a trade by a rolling 3-year z-score plus a
**volatility regime gate** (trade only when 13-week realised spread vol is in the top third of
its 3-year distribution — no movement, no opportunity).

## I.3 The agent ladder — the experimental design worth keeping

The genuinely good idea in the original project, and the one carried into the pivot:
**four agents, each seeing exactly one more piece of information.**

| Agent | Signal | Adds over previous rung |
|---|---|---|
| `agent_zero` | weekly coin flip | nothing — **null benchmark** |
| `agent_one` | 5y z-score of raw salt net flow | real storage data |
| `agent_two` | z-score of `dsi_raw` (convex → EWMA → deseasonalised) | normalisation, convexity, smoothing, seasonality |
| `agent_dsi` | z-score of `dsi_clean` + vol gate | basis residualisation + regime gating |

The gap between adjacent rungs *isolates the contribution of that one piece*. Zero exists so
that "better than nothing" has a concrete, measurable meaning.

---
# Part II — Why it didn't work

Three independent findings, in the order they were discovered. The first two were **defects in
measurement**; the third was a defect in the **hypothesis**.

## II.1 Bug one — phantom P&L at contract rolls

To backtest a spread over years you need one continuous price series. But futures expire, so
the series is **spliced** from successive contract pairs. On the splice date the quoted level
jumps, because the two pairs are *different instruments*.

Differencing across that jump books profit that no trader could capture.

In [ ]:
# A real example from the NG prompt spread:
#   2026-01-21   Feb/Mar spread = +0.673
#   2026-01-22   Mar/Apr spread = +0.093     "move" = -0.580
#
# Nothing moved. You would have CLOSED Feb/Mar and OPENED Mar/Apr.

spread = pd.Series([0.673, 0.093])          # spliced series
naive_pnl = spread.diff().iloc[-1]          # what the old backtest booked
print(f"naive P&L booked on the roll day : {naive_pnl:+.3f}   <-- fabricated")
print(f"true P&L from the switch          : {0.0:+.3f}   <-- correct")

# Measured across the real series, the damage was not marginal:
measured = pd.DataFrame({
    "instrument": ["prompt", "seasonal", "mar_apr"],
    "roll_jump_vs_normal_move": ["8.1x", "6.6x", "14.4x"],
    "share_of_ALL_movement_on_roll_days": ["31.0%", "26.8%", "5.8%"],
})
print()
print(measured.to_string(index=False))

**Roughly a quarter to a third of all "price movement" the backtest traded on was an artifact.**

The fix mirrors what the live engine already did (its delivery guard forces an exit before
expiry, so it never holds across a roll): book **zero** P&L on a roll and close the position.
`simulate(roll_flag=...)` now does this, and a regression test locks it in.

## II.2 Bug two — the backtest was running cost-free

The equity loop computed slippage, wrote it into `equity[i]` … and then **overwrote that same
cell** on the next line:

```python
equity.iloc[i] -= slippage...                             # cost computed
equity.iloc[i] = equity.iloc[i-1] + pnl_series.iloc[i]    # cost DISCARDED
```

Costs were never applied. This directly violated the locked spec — *"costs charged before
judging; a strategy that only works pre-cost is rejected."*

**The tell:** a coin flip trading 253 times showed a profit. Random trading cannot have edge,
so the measurement had to be wrong. Costs are now accumulated separately and applied in the
recursion, with a regression test asserting equity actually falls.

## II.3 The null benchmark — one seed is not a distribution

`agent_zero` used a single random seed. That is **one sample from a distribution**, not a
benchmark. Judging a signal against one coin-flip path is like calling a coin biased after one
sequence of flips.

Running 200 random agents settled it:

In [ ]:
null = pd.DataFrame({
    "instrument": ["prompt", "seasonal", "mar_apr"],
    "null_p5":    [-41.71, -31.72, -24.85],
    "null_p50":   [-10.53,  -0.34,  -5.10],
    "null_p95":   [+17.11, +24.04, +15.36],
    "seed0_used": [ -3.32, +26.31,  +5.73],
    "seed0_pctile":["69th", "95th", "78th"],
})
print(null.to_string(index=False))
print()
print("The +26.31% that looked like signal on 'seasonal' was the 95th percentile")
print("of PURE CHANCE. Not a bug, not an edge - a lucky draw.")

In [ ]:
# The decisive comparison: did ANY signal agent escape the null band?
verdict = pd.DataFrame({
    "instrument": ["prompt", "seasonal", "mar_apr"],
    "null_band_p5_p95": ["-41.7 .. +17.1", "-31.7 .. +24.0", "-24.9 .. +15.4"],
    "agent_one": [ +3.67,  -1.09,  +3.76],
    "agent_two": [ -7.10,  -5.72,  -6.01],
    "agent_dsi": [ -4.96,  -2.76,  -3.79],
})
print(verdict.to_string(index=False))
print()
print("NOT ONE signal agent, on ANY instrument, fell outside the p5-p95 band.")
print("Every result is statistically indistinguishable from random trading.")

## II.4 The week-1 regime gate — the hypothesis itself was wrong

The strategy document contained an explicit pause-gate that had never been run:

> *"Post-2016 LNG export growth tightened the deliverability system; calendar spreads have not
> fully repriced this regime. (Week-1 statistical test; if it fails, the project pauses for
> rescoping.)"*

It became testable only once the Datastream contract data (back to 1990) replaced EIA's
four-contracts-out feed. Split at 2016-02-24, the first Sabine Pass export cargo:

In [ ]:
regime = pd.DataFrame({
    "metric":     ["weekly volatility", "seasonal amplitude", "mean level",
                   "mean reversion (AR1)"],
    "pre_2016":   ["0.1932", "0.5691", "-0.472", "+0.057"],
    "post_2016":  ["0.1376", "0.5656", "-0.330", "+0.001"],
    "change":     ["-29%  (p=0.033)", "unchanged (0.99x)", "+0.14 (p=0.002)", "~vanished"],
})
print(regime.to_string(index=False))

**The premise predicted the opposite of what happened.**

A *tightening* deliverability system should produce **more** volatile spreads and **larger**
seasonal swings — scarcity shows up as violent seasonality. Instead spreads became **29% less
volatile** with **unchanged** seasonality, and mean reversion was arbitraged to zero.

That is the signature of a market becoming **more efficient**, not more stressed. Shale
abundance plausibly offset LNG demand, and more participants competed away the mean reversion.

This explains the null result coherently: **no agent beat chance because there was no
mispricing to capture.** Per the project's own rules, that triggers a pause and rescope.

---
# Part III — The pivot: cross-sectional commodity factors

## III.1 What changed, and why this is a different kind of bet

The original strategy was **one original hypothesis about one market**. If the hypothesis was
wrong, there was nothing left — which is exactly what happened.

Pivot A instead harvests premia that are **documented, replicated, and economically motivated**
across decades of literature and dozens of markets. The bet is not "I have found something
nobody else has"; it is "these well-known risk premia continue to be compensated, and I can
capture them after costs."

**Universe:** 23 USD commodity futures — energy (CL, NG, HO, RB), metals (GC, SI, HG, PL, PA),
grains/oilseeds (ZC, ZS, ZM, ZL, ZO, KE, MWE), softs (CC, KC, SB, CT, OJ), livestock (LE, GF).
8,538 contracts, 2000–2026.

**Excluded deliberately:** mini contracts (duplicate exposure), swaps and crack spreads
(derivative of the same markets — would double-count), non-USD listings (currency contamination).

## III.2 The three signals

All three are computed per market per day, then **cross-sectionally z-scored within each date**
before blending. Standardisation is essential — carry is an annualised log slope, momentum is a
12-month return. Without it, units alone would decide the portfolio.

In [ ]:
def carry(front_px, second_px, dt_years):
    '''SIGNAL 1 - the slope of the market's own forward curve.

    Positive = BACKWARDATION (front above deferred) = scarcity. Inventories are
    tight, holders of the physical earn a convenience yield, and longs are paid
    to supply the deferred month.
    Negative = CONTANGO = comfortable inventories; you pay to hold.

    Annualised because markets list on different cycles - a 1-month gap and a
    3-month gap are not comparable raw.'''
    return np.log(front_px / second_px) / dt_years

print("Backwardated (front 3.20, deferred 3.05, 1 month apart):")
print(f"   carry = {carry(3.20, 3.05, 1/12):+.4f}  -> long candidate")
print("Contangoed (front 3.05, deferred 3.20, 1 month apart):")
print(f"   carry = {carry(3.05, 3.20, 1/12):+.4f}  -> short candidate")

In [ ]:
def momentum(front_returns, window=252):
    '''SIGNAL 2 - trailing 12-month compounded return of the front contract.

    The most robust anomaly across essentially every asset class; usually
    attributed to slow information diffusion / under-reaction.

    CRITICAL: built from ROLL-SAFE returns - each return is computed between two
    prices of the SAME contract, never across a contract change. This is exactly
    the bug that corrupted the NG work (Part II.1).'''
    r = pd.Series(front_returns)
    return (1 + r).rolling(window, min_periods=window // 2).apply(np.prod, raw=True) - 1


def basis_momentum(front_returns, second_returns, window=252):
    '''SIGNAL 3 - momentum(front) minus momentum(second)  [Boons & Prado, 2019].

    Measures how the CURVE ITSELF has been moving - distinct from the curve's
    current shape (carry) and from the market's overall trend (momentum).
    Captures imbalances in intermediation of the futures curve; empirically it
    prices variation the other two miss.'''
    return momentum(front_returns, window) - momentum(second_returns, window)

rng = np.random.default_rng(0)
f = rng.normal(0.0004, 0.012, 300)       # front contract daily returns
s = f * 0.85 + rng.normal(0, 0.004, 300) # second moves similarly but not identically
print(f"12m momentum (front)  : {momentum(f).iloc[-1]:+.4f}")
print(f"12m momentum (second) : {momentum(s).iloc[-1]:+.4f}")
print(f"basis-momentum        : {basis_momentum(f, s).iloc[-1]:+.4f}  <- curve dynamics")

## III.3 The agent ladder, rebuilt

Identical experimental logic to the original — each rung sees exactly one more factor:

| Agent | Sees | Role |
|---|---|---|
| `agent_0` | **nothing** — seeded random score per market | **Null benchmark**, run across many seeds to form a *distribution* |
| `agent_1` | carry | first real signal |
| `agent_2` | carry + momentum | adds trend |
| `agent_3` | carry + momentum + basis-momentum | **full signal** |

Score = **equal-weight mean of the standardised factors that rung can see**. No factor weights
are fitted — a deliberate choice to avoid in-sample optimisation.

In [ ]:
def cross_sectional_zscore(values):
    '''Standardise within a single date, so heterogeneous factors are comparable
    and the portfolio depends on RELATIVE rankings - which is what a
    dollar-neutral long/short book actually expresses.'''
    v = pd.Series(values)
    return (v - v.mean()) / v.std()

# One rebalance date, 9 markets, agent_3's three factors:
mkts = ["CL","NG","GC","HG","ZC","ZS","KC","SB","LE"]
raw = pd.DataFrame({
    "carry":     [ 0.12,-0.30, -0.05, 0.02,-0.15, 0.08, 0.22,-0.10, 0.04],
    "mom":       [ 0.18, 0.05, 0.11,-0.07, 0.02,-0.12, 0.25, 0.09,-0.03],
    "basis_mom": [ 0.03,-0.08,  0.01, 0.06,-0.02, 0.04, 0.05,-0.06, 0.02],
}, index=mkts)

z = raw.apply(cross_sectional_zscore)
score = z.mean(axis=1).sort_values(ascending=False)
print("agent_3 score (higher = want to be long):\n")
print(score.round(3).to_string())

k = 3                                        # tercile of 9 markets
print(f"\nLONG  (top {k}) : {list(score.head(k).index)}")
print(f"SHORT (bot {k}) : {list(score.tail(k).index)}")

## III.4 Portfolio construction

| Element | Choice | Why |
|---|---|---|
| Positions | long top tercile, short bottom tercile | standard factor construction |
| Weighting | equal within each side; each side sums to 1.0 gross | no fitted weights |
| Neutrality | **dollar-neutral** (+1 long / −1 short, net 0) | strips out common commodity beta so what remains is the *factor*, not a bet on commodities rising |
| Breadth floor | ≥ 6 markets (≥ 2 per side) | thinner cross-sections are noise |
| Rebalance | month-end, held between | keeps turnover realistic |
| Look-ahead guard | weights set at close, `.shift(1)` before earning | a position cannot earn the return that selected it |

**Volatility targeting** (10% annualised) scales exposure by `target_vol / trailing_vol`, using a
63-day estimate **shifted one day** so there is no look-ahead, capped at 3×. In practice average
leverage is **0.53** — the rule mostly *de-risks*.

**Costs**: 10 bps per 1.0 of gross weight traded, charged on turnover, and stress-tested at 3×
and 5×. Deliberately conservative for liquid futures (real round-trip ≈ 2–5 bps).

---
# Part IV — Results

## IV.1 The ladder (2000–2026, 6,676 trading days)

In [ ]:
ladder = pd.DataFrame({
    "agent":   ["agent_0","agent_1","agent_2","agent_3"],
    "signal":  ["random (null)","carry","carry+momentum","carry+mom+basis-mom"],
    "ann_ret%":[  -0.26,   2.91,   5.56,  11.54],
    "sharpe":  [   0.08,   0.24,   0.36,   0.61],
    "max_dd%": [ -66.3,  -75.0,  -63.8,  -36.4],
    "vs_chance":["--","98th pctile","100th","100th"],
})
print(ladder.to_string(index=False))
print("\nMonotonic: every rung adds, and every signal rung beats the null band.")

## IV.2 Robustness — the tests that killed the NG result

| Test | Result |
|---|---|
| **5× costs** | `agent_3` 0.61 → **0.52**. Not cost-fragile. |
| **Null sanity** | random earns Sharpe **+0.02 at zero cost** — properly zero, confirming the simulator manufactures nothing |
| **Turnover artifact?** | at zero cost for both: `agent_3` 0.64 vs random p95 0.28. The edge is prediction, not trading less |
| **Vol targeting** | Sharpe 0.61 → **0.70**, max DD −36% → **−19%** |
| **Post-publication decay** | 0.59 (pre-2019) → 0.69 (post-2019) → **0.93** (last 3y). No decay |
| **Hardest test** (post-2019 + 3× costs + vol-targeted) | **Sharpe 0.78, +8.2%/yr, −19.3% DD** |

## IV.3 The carry paradox — and why a pre-committed test mattered

Carry's **standalone** Sharpe has decayed to negative: **−0.51 over the last 3 years**. The
premium has plausibly been arbitraged since commodity index investing took off post-2005.

That made "drop carry" look obvious. A **pre-committed, single-shot** test said otherwise:

In [ ]:
# Actual output of scripts/carry_drop_test.py (run once, rule fixed in advance)
carry_test = pd.DataFrame({
    "condition":      ["full sample","post-2019","vol-targeted","post-2019 + 3x + volT"],
    "with_carry":     [ 0.612, 0.690, 0.698, 0.784],
    "without_carry":  [ 0.430, 0.650, 0.442, 0.743],
})
carry_test["delta"] = carry_test["without_carry"] - carry_test["with_carry"]
print(carry_test.round(3).to_string(index=False))
print("\nimprovements: 0 of 4  (threshold to drop was 3 of 4)")
print("DECISION: KEEP CARRY.")

**Why standalone Sharpe was the wrong criterion.** Carry is weakly correlated with momentum and
basis-momentum, so it **reduces portfolio variance more than it reduces return**. A factor can
be a poor standalone bet and still earn its place through diversification.

The correct question is *marginal contribution to the portfolio*, not standalone performance.

This is also a methodology point: the test was specified in advance with a fixed decision rule
and run **once**. Had it been run iteratively — trying variations until a number improved — a
genuine benefit would likely have been discarded, and whatever survived would have been fitted
to noise.

---
# Part V — Status and honest limitations

## What is verified
- Monotonic ladder; every signal rung beats a calibrated null
- Survives 5× costs; no post-publication decay; vol targeting halves drawdown
- Roll-safe by construction — the phantom-P&L bug **cannot** recur
- Costs explicitly charged, with regression tests on both former bugs

## What is not
1. **No out-of-sample holdout.** Universe, 12-month momentum window, tercile cutoffs, monthly
   rebalance, 10% vol target and 63-day vol window were all chosen by judgement. Nothing was
   tuned iteratively — but that is not the same as validated.
2. **Post-2019 sample is short** (~7.5 yrs) with a wide null band (sd 0.33).
3. **Datastream settlements unverified** against an independent source.
4. **Flat cost model** — oats and orange juice are not crude oil. No liquidity filtering yet.
5. **Execution not modelled** — 23 single-contract positions is a different problem from one
   calendar spread: per-market contract sizing, whole-lot rounding, margin, and market-data
   entitlements all remain open.

## Next
1. Freeze the specification into `config/` so it is auditable and cannot drift.
2. **Forward paper trading** — the only test that cannot be overfit. Run all four agents so
   `agent_0` provides a live forward null.
3. Retire the frozen NG books cleanly.

> **The core lesson.** Two measurement bugs and one untested premise produced a result that
> looked, at various moments, like a monotonic information ladder and a +26% strategy. Neither
> was real. What found the errors was not cleverness but discipline: a null benchmark that
> should earn zero, costs that must be charged, and returns that may only be computed between
> two prices of the same contract.